In [1]:
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

In [2]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Convert y-z-value CSV to OpenFOAM interpolation2DTable format."
    )
    parser.add_argument("--input", required=True, help="Input CSV file")
    parser.add_argument("--output", required=True, help="Output OpenFOAM table file")
    parser.add_argument(
        "--mode",
        choices=["resample", "bin"],
        default="resample",
        help="Conversion mode: 'resample' for scattered data, 'bin' for already structured data",
    )

    # Column indices in the CSV file (0-based)
    parser.add_argument("--y-col", type=int, default=0, help="Column index for y")
    parser.add_argument("--z-col", type=int, default=1, help="Column index for z")
    parser.add_argument("--value-col", type=int, default=2, help="Column index for value")

    # Resampling options
    parser.add_argument("--ny", type=int, default=51, help="Number of target y grid points")
    parser.add_argument("--nz", type=int, default=101, help="Number of target z grid points")
    parser.add_argument(
        "--fill-nearest",
        action="store_true",
        help="Use nearest-neighbor values where linear interpolation returns NaN",
    )

    # Direct binning options
    parser.add_argument(
        "--round-decimals",
        type=int,
        default=3,
        help="Number of decimals used to group y and z in 'bin' mode",
    )

    # Output formatting
    parser.add_argument(
        "--fmt",
        default=".8g",
        help="Python format specifier for written numbers, e.g. '.8g' or '.6f'",
    )
    return parser.parse_args()

In [3]:
def read_csv_columns(csv_path: str, y_col: int, z_col: int, value_col: int) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    cols = df.columns.tolist()

    max_index = max(y_col, z_col, value_col)
    if max_index >= len(cols):
        raise IndexError(
            f"CSV has {len(cols)} columns, but requested index {max_index}."
        )

    out = pd.DataFrame(
        {
            "y": pd.to_numeric(df.iloc[:, y_col], errors="coerce"),
            "z": pd.to_numeric(df.iloc[:, z_col], errors="coerce"),
            "value": pd.to_numeric(df.iloc[:, value_col], errors="coerce"),
        }
    ).dropna()

    return out.sort_values(["y", "z"]).reset_index(drop=True)

In [4]:
def build_table_by_binning(df: pd.DataFrame, round_decimals: int) -> list[tuple[float, list[tuple[float, float]]]]:
    tmp = df.copy()
    tmp["y_bin"] = tmp["y"].round(round_decimals)
    tmp["z_bin"] = tmp["z"].round(round_decimals)

    # Average values that collapse onto the same rounded (y, z) location.
    grouped = (
        tmp.groupby(["y_bin", "z_bin"], as_index=False)["value"]
        .mean()
        .sort_values(["y_bin", "z_bin"])
    )

    table: list[tuple[float, list[tuple[float, float]]]] = []
    for y_val, sub in grouped.groupby("y_bin", sort=True):
        yz_list = [(float(z), float(v)) for z, v in zip(sub["z_bin"], sub["value"])]
        if len(yz_list) >= 2:
            table.append((float(y_val), yz_list))

    return table

In [5]:
def build_table_by_resampling(
    df: pd.DataFrame,
    ny: int,
    nz: int,
    fill_nearest: bool,
) -> list[tuple[float, list[tuple[float, float]]]]:
    points = df[["y", "z"]].to_numpy()
    values = df["value"].to_numpy()

    y_min, y_max = df["y"].min(), df["y"].max()
    z_min, z_max = df["z"].min(), df["z"].max()

    y_grid = np.linspace(y_min, y_max, ny)
    z_grid = np.linspace(z_min, z_max, nz)

    linear_interp = LinearNDInterpolator(points, values, fill_value=np.nan)
    nearest_interp = NearestNDInterpolator(points, values) if fill_nearest else None

    table: list[tuple[float, list[tuple[float, float]]]] = []

    for y_val in y_grid:
        yz_list: list[tuple[float, float]] = []
        query_points = np.column_stack([np.full_like(z_grid, y_val), z_grid])
        vals = linear_interp(query_points)

        if fill_nearest and nearest_interp is not None:
            nan_mask = np.isnan(vals)
            if np.any(nan_mask):
                vals[nan_mask] = nearest_interp(query_points[nan_mask])

        # Keep only valid points. For a circular/irregular domain this naturally
        # truncates each y-slice to the local z-range that is covered by the data.
        valid_mask = ~np.isnan(vals)
        valid_z = z_grid[valid_mask]
        valid_v = vals[valid_mask]

        if len(valid_z) >= 2:
            yz_list = [(float(z), float(v)) for z, v in zip(valid_z, valid_v)]
            table.append((float(y_val), yz_list))

    return table

In [6]:
def write_interpolation2dtable(
    output_path: str,
    table: list[tuple[float, list[tuple[float, float]]]],
    number_fmt: str,
) -> None:
    def fmt(x: float) -> str:
        return format(float(x), number_fmt)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("// -*- C++ -*-\n")
        f.write("(\n")
        for y_val, yz_list in table:
            f.write("    (\n")
            f.write(f"        {fmt(y_val)}\n")
            f.write("        (\n")
            for z_val, v_val in yz_list:
                f.write(f"            ({fmt(z_val)} {fmt(v_val)})\n")
            f.write("        )\n")
            f.write("    )\n")
        f.write(")\n")

In [10]:
def main() -> None:
    args = parse_args()
    df = read_csv_columns(args.input, args.y_col, args.z_col, args.value_col)

    if args.mode == "bin":
        table = build_table_by_binning(df, round_decimals=args.round_decimals)
    else:
        table = build_table_by_resampling(
            df,
            ny=args.zny,
            nz=args.nz,
            fill_nearest=args.fill_nearest,
        )

    if not table:
        raise RuntimeError("No valid table entries were generated.")

    write_interpolation2dtable(args.output, table, args.fmt)

    print(f"Wrote {len(table)} outer y-slices to: {Path(args.output).resolve()}")
    print("First y-slice contains", len(table[0][1]), "(z, value) pairs.")
    print("Last  y-slice contains", len(table[-1][1]), "(z, value) pairs.")


# if __name__ == "__main__":
#     main()

```
python3 csv_to_interpolation2DTable.py \
    --input data_U.csv \
    --output UxProfile2D.table \
    --mode bin \
    --round-decimals 3
```